# Magic Formula en EE. UU. (2001–2026): EBIT histórico frente a EBIT previsto por los analistas

**Pregunta:** ¿funciona mejor la Magic Formula si, en lugar del EBIT de los últimos 12 meses (Greenblatt), se usa el EBIT que preveían los analistas en cada fecha?

**Datos:**
- **CRSP y Compustat (WRDS):** rentabilidades, capitalización, balances y EBIT de los últimos 12 meses, con la fecha de publicación de cada dato.
- **Bloomberg (BEst):** consenso de EBIT a 12 meses de las 1.092 empresas que han pasado por el S&P 500, a 31 de diciembre de cada año (archivo `bloomberg_forward_eeuu.csv`).

**Antes de empezar:** sube `bloomberg_forward_eeuu.csv` a la misma carpeta que este notebook (panel de archivos de la izquierda → flecha hacia arriba, *Upload*). Luego ejecuta todas las celdas en orden con `Shift + Enter` (o menú *Run → Run All Cells*).

> Datos de WRDS y Bloomberg para uso académico. Solo se guardan resultados agregados.

## 1. Parámetros del estudio
Aquí se cambian las opciones sin tocar el resto del código.

In [ ]:
ANIOS = list(range(2001, 2027))   # 1 de enero de cada año. 2026 es el año en curso (parcial)
TOP_N = 30                        # nº de empresas de la cartera (Greenblatt: 20-30)
ROC_METODO = "tangible"           # "tangible" = Greenblatt; "gw" = sumando fondo de comercio e intangibles
UNIVERSO = "auto"                 # "auto": S&P 500 histórico solo si está completo; "mayores": las N mayores
TAMANO_UNIVERSO = 500             # tamaño del universo de las mayores empresas
USUARIO_WRDS = "jaimeandres"

INICIO_DATOS = "1998-01-01"       # margen para tener 4 trimestres de EBIT en 2001
FIN_DATOS = "2026-12-31"

## 2. Conexión con WRDS
Te pedirá la contraseña (escríbela tú) y puede que tengas que aprobar el aviso de **Duo** en el móvil.

In [ ]:
import wrds
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 20)
import warnings
# Python 3.14 confunde el detector de "chained assignment" de pandas y da falsas alarmas
warnings.filterwarnings("ignore", message=".*[Cc]hained assignment.*")
warnings.filterwarnings("ignore", category=FutureWarning)

db = wrds.Connection(wrds_username=USUARIO_WRDS)

## 3. Localizar las tablas
CRSP tiene dos formatos: el antiguo (`msf`) y el nuevo de 2024 (`msf_v2`). Esta celda detecta cuál tiene CUNEF y qué columnas usa, y descarga el **enlace entre CRSP y Compustat** (qué acción de CRSP corresponde a cada empresa de Compustat).

In [ ]:
def columnas(lib, tabla):
    return set(db.describe_table(library=lib, table=tabla)["name"].str.lower())

def elegir(cols, *opciones):
    for o in opciones:
        if o in cols:
            return o
    raise KeyError(f"No encuentro ninguna de {opciones}. Columnas disponibles: {sorted(cols)}")

def sql_lista(valores):
    # Lista para una consulta SQL: (10104, 10107) o ('001004', '001045'), sin tipos de numpy
    limpios = sorted({int(v) if isinstance(v, (int, np.integer)) else str(v) for v in valores})
    return "(" + ", ".join(str(v) if isinstance(v, int) else "'" + v.replace("'", "''") + "'" for v in limpios) + ")"

def intentar(sql, date_cols=None, aviso=""):
    # Ejecuta una consulta; si no hay permiso o no existe la tabla, devuelve None en vez de parar
    try:
        return db.raw_sql(sql, date_cols=date_cols)
    except Exception as e:
        print(f"   · {aviso}: no disponible ({str(e).splitlines()[0][:100]})")
        return None

tablas_crsp = set(db.list_tables(library="crsp"))

# Rentabilidades mensuales
if "msf_v2" in tablas_crsp:
    MSF = "msf_v2"
    c = columnas("crsp", MSF)
    COL_FECHA, COL_RET = elegir(c, "mthcaldt", "date"), elegir(c, "mthret", "ret")
    COL_PRC, COL_SHR = elegir(c, "mthprc", "prc"), elegir(c, "shrout")
    COLS_TIPO = [x for x in ["securitytype", "securitysubtype", "sharetype"] if x in c]
    BAJAS_INCLUIDAS = True      # en el formato nuevo la rentabilidad de salida ya va incluida
else:
    MSF, COL_FECHA, COL_RET, COL_PRC, COL_SHR = "msf", "date", "ret", "prc", "shrout"
    COLS_TIPO, BAJAS_INCLUIDAS = [], False
print(f"Rentabilidades: crsp.{MSF} ({COL_FECHA}, {COL_RET}, {COL_PRC}, {COL_SHR})")

# Enlace CRSP-Compustat
link = None
for tabla in ["crsp.ccmxpf_lnkhist", "crsp_a_ccm.ccmxpf_lnkhist"]:
    link = intentar(f'''
        select gvkey, lpermno as permno, linkdt, linkenddt from {tabla}
        where linktype in ('LU', 'LC') and linkprim in ('P', 'C')
    ''', ["linkdt", "linkenddt"], tabla)
    if link is not None:
        print(f"Enlace CRSP-Compustat: {tabla}")
        break
if link is None:
    raise RuntimeError("Sin acceso al enlace CRSP-Compustat. Hay que preguntar a la biblioteca de CUNEF.")
link = link.dropna(subset=["permno"]).copy()
link["permno"] = link.permno.astype(int)
link["linkenddt"] = link.linkenddt.fillna(pd.Timestamp("2099-12-31"))

## 4. El universo de empresas y los datos
**Qué empresas entran cada año.** Lo ideal es la composición histórica oficial del S&P 500. Esta celda la busca en CRSP y en Compustat. Si la suscripción de CUNEF no la incluye, construye un universo equivalente: **cada mes, las 500 mayores empresas de EE. UU. por capitalización** (solo acciones ordinarias con ficha en Compustat, sin fondos). Coincide casi del todo con el S&P 500, también evita el sesgo de supervivencia y es un método habitual en estudios académicos.

Después descarga las rentabilidades, los fundamentales y los sectores. Puede tardar unos minutos.

In [ ]:
# 4.1 Composición histórica del S&P 500, si hay acceso
sp, FUENTE_UNIVERSO = None, None
for tabla in ["dsp500list_v2", "dsp500list"]:
    if sp is None and tabla in tablas_crsp:
        try:
            c = columnas("crsp", tabla)
            c_ini, c_fin = elegir(c, "mbrstartdt", "start"), elegir(c, "mbrenddt", "ending")
            sp = intentar(f"select permno, {c_ini} as ini, {c_fin} as fin from crsp.{tabla} "
                          f"where {c_fin} >= '{INICIO_DATOS}'", ["ini", "fin"], f"crsp.{tabla}")
            if sp is not None:
                FUENTE_UNIVERSO = f"S&P 500 histórico (crsp.{tabla})"
        except Exception as e:
            print(f"   · crsp.{tabla}: no disponible ({str(e).splitlines()[0][:100]})")
if sp is None:
    idx = intentar(f'''
        select gvkey, "from" as ini, thru as fin from comp.idxcst_his
        where gvkeyx = '000003' and (thru is null or thru >= '{INICIO_DATOS}')
    ''', ["ini", "fin"], "comp.idxcst_his")
    if idx is not None and len(idx):
        sp = idx.merge(link[["gvkey", "permno"]].drop_duplicates(), on="gvkey")[["permno", "ini", "fin"]]
        FUENTE_UNIVERSO = "S&P 500 histórico (comp.idxcst_his)"

# Comprobar que la lista histórica está completa: el S&P 500 tiene ~500 acciones todos los años.
# Si faltan las empresas que salieron del índice, usarla introduciría sesgo de supervivencia.
if sp is not None:
    sp["fin"] = sp.fin.fillna(pd.Timestamp("2099-12-31"))
    conteos = {a: sp[(sp.ini <= pd.Timestamp(f"{a}-01-01")) &
                     (sp.fin >= pd.Timestamp(f"{a}-01-01"))].permno.nunique() for a in range(2015, 2026)}
    print(f"Acciones por año en {FUENTE_UNIVERSO}: {conteos}")
    completa = min(conteos.values()) >= 0.94 * TAMANO_UNIVERSO and max(conteos.values()) <= 1.08 * TAMANO_UNIVERSO
    if UNIVERSO == "mayores" or not completa:
        if not completa:
            print("   · La lista está INCOMPLETA (debería haber ~500 cada año): le faltan empresas que salieron "
                  "del índice. Para evitar el sesgo de supervivencia, uso las mayores empresas de CRSP.")
        sp, FUENTE_UNIVERSO = None, None

USA_SP500 = sp is not None
if USA_SP500:
    sp["permno"] = sp.permno.astype(int)
    sp["fin"] = sp.fin.fillna(pd.Timestamp("2099-12-31"))
    NOMBRE_INDICE = "S&P 500"
else:
    FUENTE_UNIVERSO = f"las {TAMANO_UNIVERSO} mayores empresas de EE. UU. por capitalización (CRSP)"
    NOMBRE_INDICE = f"{TAMANO_UNIVERSO} mayores de EE. UU."
print(f"\nUniverso utilizado: {FUENTE_UNIVERSO}\n")

# 4.2 Rentabilidades mensuales (con dividendos) y capitalización
filtro = f"and permno in {sql_lista(sp.permno.unique())}" if USA_SP500 else ""
extra = "".join(f", {x}" for x in COLS_TIPO)
msf = db.raw_sql(f'''
    select permno, {COL_FECHA} as fecha, {COL_RET} as ret, {COL_PRC} as prc, {COL_SHR} as shrout{extra}
    from crsp.{MSF}
    where {COL_FECHA} between '{INICIO_DATOS}' and '{FIN_DATOS}' {filtro}
''', date_cols=["fecha"])
msf["permno"] = msf.permno.astype(int)
msf["mes"] = msf.fecha.dt.to_period("M")

if not BAJAS_INCLUIDAS:
    # Formato antiguo: añadir la rentabilidad del día en que la acción deja de cotizar
    bajas = intentar(f"select permno, dlstdt, dlret from crsp.msedelist where dlstdt >= '{INICIO_DATOS}' {filtro}",
                     ["dlstdt"], "crsp.msedelist")
    if bajas is not None:
        bajas["permno"] = bajas.permno.astype(int)
        bajas["mes"] = bajas.dlstdt.dt.to_period("M")
        msf = msf.merge(bajas[["permno", "mes", "dlret"]], on=["permno", "mes"], how="left")
        msf["ret"] = np.where(msf.dlret.notna(), (1 + msf.ret.fillna(0)) * (1 + msf.dlret) - 1, msf.ret)
        msf = msf.drop(columns=["dlret"])

msf["mcap"] = msf.prc.abs() * msf.shrout

# 4.3 Universo de las mayores empresas (solo si no hay lista histórica del S&P 500)
if not USA_SP500:
    elegibles = msf[msf.permno.isin(set(link.permno))]          # con ficha en Compustat: excluye fondos
    if COLS_TIPO:
        comun = pd.Series(True, index=elegibles.index)
        if "securitytype" in elegibles:
            comun &= elegibles.securitytype.eq("EQTY")
        if "securitysubtype" in elegibles:
            comun &= elegibles.securitysubtype.eq("COM")
        if "sharetype" in elegibles:
            comun &= elegibles.sharetype.eq("NS")             # excluye ADR y otros tipos
        if comun.mean() > 0.2:
            elegibles = elegibles[comun]
        else:
            print("   · aviso: el filtro de acciones ordinarias descartaría casi todo; no se aplica")
    else:
        nombres = intentar("select distinct permno from crsp.msenames where shrcd in (10, 11, 12)",
                           aviso="crsp.msenames")
        if nombres is not None:
            elegibles = elegibles[elegibles.permno.isin(set(nombres.permno.astype(int)))]
    top = (elegibles.dropna(subset=["mcap"])
                    .sort_values(["mes", "mcap"], ascending=[True, False])
                    .groupby("mes").head(TAMANO_UNIVERSO + 100))     # candidatos; se depuran en 4.6
    MIEMBROS_POR_MES = top.groupby("mes").permno.apply(lambda x: set(x.astype(int))).to_dict()


def miembros_en(fecha):
    # Empresas del universo en una fecha (con la capitalización del mes anterior, sin mirar al futuro)
    if USA_SP500:
        return sp[(sp.ini <= fecha) & (sp.fin >= fecha)].permno.unique()
    return list(MIEMBROS_POR_MES.get((fecha - pd.Timedelta(days=1)).to_period("M"), set()))


permnos_universo = set(sp.permno) if USA_SP500 else set().union(*MIEMBROS_POR_MES.values())
link = link[link.permno.isin(permnos_universo)].copy()
lista_gvkeys = sql_lista(link.gvkey.unique())
print(f"Empresas que han pasado por el universo desde 2012: {link.gvkey.nunique()}")

# 4.4 Fundamentales trimestrales (en millones de $) con fecha de publicación
fq = db.raw_sql(f'''
    select gvkey, datadate, rdq, curcdq, oiadpq, actq, lctq, cheq, dlcq, dlttq, ppentq, pstkq, mibq, intanq
    from comp.fundq
    where indfmt = 'INDL' and datafmt = 'STD' and popsrc = 'D' and consol = 'C'
      and datadate between '{INICIO_DATOS}' and '{FIN_DATOS}' and gvkey in {lista_gvkeys}
''', date_cols=["datadate", "rdq"])
fq = fq.sort_values(["gvkey", "datadate"]).drop_duplicates(["gvkey", "datadate"], keep="last")
# Si falta la fecha de publicación, asumimos 90 días después del cierre del trimestre (prudente)
fq["disponible"] = fq.rdq.fillna(fq.datadate + pd.Timedelta(days=90))

# 4.5 Nombre y sector de cada empresa
comp = db.raw_sql(f"select gvkey, conm, gsector, sic from comp.company where gvkey in {lista_gvkeys}")

# 4.6 Solo empresas que presentan sus cuentas en dólares. Algunas extranjeras cotizan en EE. UU.
#     (p. ej. CGI en dólares canadienses o ASML en euros): mezclar su EBIT en otra divisa con su
#     capitalización en dólares falsearía el Earnings Yield.
moneda = fq.dropna(subset=["curcdq"]).groupby("gvkey").curcdq.agg(lambda x: x.mode().iat[0])
gvkeys_no_usd = set(moneda[moneda != "USD"].index)
permnos_no_usd = set(link[link.gvkey.isin(gvkeys_no_usd)].permno)
if not USA_SP500:
    top = top[~top.permno.isin(permnos_no_usd)].groupby("mes").head(TAMANO_UNIVERSO)
    MIEMBROS_POR_MES = top.groupby("mes").permno.apply(lambda x: set(x.astype(int))).to_dict()
print(f"Excluidas por presentar cuentas en otra divisa: {len(gvkeys_no_usd)} empresas "
      f"(p. ej. {', '.join(comp[comp.gvkey.isin(gvkeys_no_usd)].conm.head(5))})")

print(f"Rentabilidades: {len(msf):,} filas | Fundamentales: {len(fq):,} trimestres | Empresas: {len(comp):,}")

## 5. Comprobaciones de calidad
Antes de calcular nada, compruebo que los datos tienen sentido. Un estudio profesional siempre hace esto.

In [ ]:
# Unidades de la capitalización: las 500 mayores de EE. UU. valen entre 15 y 60 billones de $ en este periodo
dentro = miembros_en(pd.Timestamp("2025-01-01"))
ref = pd.Period("2024-12", "M")
total = msf[(msf.mes == ref) & msf.permno.isin(dentro)].mcap.sum()
msf["mcap"] = msf.mcap / (1e6 if total > 1e12 else 1e3)        # pasar a millones de $
total_m = msf[(msf.mes == ref) & msf.permno.isin(dentro)].mcap.sum()
print(f"Capitalización total del universo ({NOMBRE_INDICE}) en dic-2024: "
      f"{total_m / 1e6:,.1f} billones de $  (esperado: ~45-55)")

# Último mes con datos completos
cobertura = msf.dropna(subset=["ret"]).groupby("mes").permno.nunique()
ULTIMO_MES = cobertura[cobertura >= 0.8 * cobertura.max()].index.max()
print(f"Último mes con datos de rentabilidad: {ULTIMO_MES}")

for a in [2015, 2020, 2025]:
    print(f"Empresas en el universo el 1/1/{a}: {len(miembros_en(pd.Timestamp(f'{a}-01-01')))}")

## 5 bis. EBIT previsto de Bloomberg
Se une cada empresa de Bloomberg con Compustat por su código **CUSIP**. El consenso a 31 de diciembre se usa en el ranking del 1 de enero siguiente: es un dato que ya se conocía ese día, así que no hay sesgo de anticipación.

In [ ]:
bbg = pd.read_csv("bloomberg_forward_eeuu.csv", parse_dates=["fecha"])
bbg["cusip8"] = bbg.cusip.astype(str).str[:8]
cus = sql_lista(bbg.cusip8.dropna().unique())

# Compustat: CUSIP de cada emisión
sec = intentar(f"select distinct gvkey, substr(cusip, 1, 8) as cusip8 from comp.security where substr(cusip, 1, 8) in {cus}",
               aviso="comp.security")
mapa = sec if sec is not None else pd.DataFrame(columns=["gvkey", "cusip8"])
# CRSP: CUSIP histórico -> permno -> gvkey, para empresas que cambiaron de código
nom = None
for tabla, col in [("crsp.stocknames", "ncusip"), ("crsp.msenames", "ncusip"), ("crsp.stocknames_v2", "hdrcusip")]:
    nom = intentar(f"select distinct permno, {col} as cusip8 from {tabla} where {col} in {cus}", aviso=tabla)
    if nom is not None and len(nom):
        break
if nom is not None and len(nom):
    nom["permno"] = nom.permno.astype(int)
    extra = nom.merge(link[["permno", "gvkey"]].drop_duplicates(), on="permno")[["gvkey", "cusip8"]]
    mapa = pd.concat([mapa, extra]).drop_duplicates()
bbg = bbg.merge(mapa.drop_duplicates("cusip8"), on="cusip8", how="left")
print(f"Empresas de Bloomberg: {bbg.ticker.nunique()} | enlazadas con Compustat: {bbg.dropna(subset=['gvkey']).ticker.nunique()}")
bbg = bbg.dropna(subset=["gvkey"])


def ebit_previsto_en(fecha):
    # Consenso vigente el día anterior a 'fecha' (31 de diciembre), por empresa
    d = bbg[(bbg.fecha < fecha) & (bbg.fecha >= fecha - pd.Timedelta(days=400))]
    return d.sort_values("fecha").groupby("gvkey").ebit_fwd.last()


## 6. El ranking de la Magic Formula en una fecha (sin mirar al futuro)
Para cada fecha:
1. Empresas que estaban en el universo ese día y su capitalización a cierre del mes anterior.
2. Para cada empresa, los **4 últimos trimestres publicados antes de esa fecha**: EBIT de 12 meses (`oiadpq`) y el último balance.
3. Se excluyen financieras, utilities e inmobiliarias (sectores GICS 40, 55 y 60).
4. **EV** = capitalización + deuda (corto y largo plazo) + preferentes + minoritarios − caja.
5. **Capital invertido** = capital circulante neto (mínimo cero) + inmovilizado material neto (+ intangibles en la variante `gw`).
6. Se ordena por Earnings Yield y por ROC, se suman los dos puestos y gana la suma más baja.

In [ ]:
CAMPOS_BALANCE = ["actq", "lctq", "cheq", "dlcq", "dlttq", "ppentq", "pstkq", "mibq", "intanq"]

def foto_fundamental(fecha):
    # Últimos datos publicados antes de 'fecha' para cada empresa
    d = fq[(fq.disponible < fecha) & (fq.datadate >= fecha - pd.Timedelta(days=550))
           & (fq.curcdq.fillna("USD") == "USD")]
    filas = []
    for gvkey, g in d.groupby("gvkey"):
        g = g.tail(4)
        if len(g) < 4 or g.oiadpq.isna().any():
            continue                                         # sin 4 trimestres completos
        if (g.datadate.iloc[-1] - g.datadate.iloc[0]).days > 300:
            continue                                         # trimestres no consecutivos
        ult = g.iloc[-1]
        if (fecha - ult.datadate).days > 270:
            continue                                         # balance demasiado antiguo
        filas.append({"gvkey": gvkey, "ebit": g.oiadpq.sum(), "ultimo_trimestre": ult.datadate,
                      **{k: ult[k] for k in CAMPOS_BALANCE}})
    return pd.DataFrame(filas, columns=["gvkey", "ebit", "ultimo_trimestre", *CAMPOS_BALANCE])


def ranking(fecha, roc_metodo=ROC_METODO, modo="ltm"):
    # modo="ltm": EBIT de los últimos 12 meses (Greenblatt) · modo="fwd": EBIT previsto por los analistas
    miembros = miembros_en(fecha)
    mes_anterior = (fecha - pd.Timedelta(days=1)).to_period("M")
    cap = msf[(msf.mes == mes_anterior) & msf.permno.isin(miembros)][["permno", "mcap"]].dropna()
    enlaces = link[(link.linkdt <= fecha) & (link.linkenddt >= fecha)][["gvkey", "permno"]]
    cap = cap.merge(enlaces, on="permno")
    # Empresas con dos clases de acciones (p. ej. Alphabet): sumar capitalización, quedarse con la mayor
    cap = (cap.sort_values("mcap", ascending=False)
              .groupby("gvkey").agg(permno=("permno", "first"), mcap=("mcap", "sum")).reset_index())

    df = cap.merge(foto_fundamental(fecha), on="gvkey").merge(comp, on="gvkey", how="left")

    sic = pd.to_numeric(df.sic, errors="coerce")
    excluida = df.gsector.isin(["40", "55", "60"]) | (
        df.gsector.isna() & (sic.between(6000, 6999) | sic.between(4900, 4999)))
    df = df[~excluida].copy()

    z = lambda s: s.fillna(0)
    df["ev"] = df.mcap + z(df.dlttq) + z(df.dlcq) + z(df.pstkq) + z(df.mibq) - z(df.cheq)
    df["capital_circulante"] = (df.actq - z(df.cheq)) - (df.lctq - z(df.dlcq))
    df["capital"] = df.capital_circulante.clip(lower=0).fillna(0) + z(df.ppentq)
    if roc_metodo == "gw":
        df["capital"] = df.capital + z(df.intanq)
    if modo == "fwd":
        df["ebit_ltm"] = df.ebit
        df["ebit"] = df.gvkey.map(ebit_previsto_en(fecha))
    df["ey"] = df.ebit / df.ev
    df["roc"] = df.ebit / df.capital
    df = df[(df.ebit > 0) & (df.ev > 0) & (df.capital > 0)].copy()

    df["rank_ey"] = df.ey.rank(ascending=False, method="first")
    df["rank_roc"] = df.roc.rank(ascending=False, method="first")
    df["puntos"] = df.rank_ey + df.rank_roc
    df = df.sort_values(["puntos", "ey"], ascending=[True, False]).reset_index(drop=True)
    df["puesto"] = df.index + 1
    return df

### Prueba: el ranking del 1 de enero de 2025, con las dos versiones

In [ ]:
for modo in ["ltm", "fwd"]:
    p = ranking(pd.Timestamp("2025-01-01"), modo=modo)
    print(f"{modo.upper()}: {len(p)} empresas elegibles")
    display(p[["puesto", "conm", "mcap", "ebit", "ev", "ey", "roc"]].head(10))


## 7. Rentabilidad de la cartera y de los índices
- **Cartera**: mismo importe en cada empresa el 1 de enero y se mantiene todo el año (sin rebalancear). Si una empresa deja de cotizar, su dinero queda en liquidez.
- **S&P 500 ponderado**: cada mes, los miembros ponderados por su capitalización del mes anterior (equivale al índice con dividendos).
- **S&P 500 a partes iguales**: los mismos miembros, con el mismo peso. Es la comparación más justa, porque la cartera también pondera igual.

In [ ]:
def meses_del_anio(anio):
    return [m for m in pd.period_range(f"{anio}-01", f"{anio}-12", freq="M") if m <= ULTIMO_MES]


def rentabilidad_cartera(permnos, anio):
    meses = meses_del_anio(anio)
    r = (msf[msf.permno.isin(permnos) & msf.mes.isin(meses)]
         .pivot_table(index="mes", columns="permno", values="ret")
         .reindex(index=meses, columns=permnos))
    valor = (1 + r.fillna(0)).cumprod()          # valor de cada posición (sin datos = liquidez)
    cartera = valor.mean(axis=1)                 # mismo peso inicial
    return cartera / cartera.shift(1, fill_value=1.0) - 1


def rentabilidad_indice(anio, ponderado=True):
    salida = {}
    for m in meses_del_anio(anio):
        miembros = miembros_en(m.start_time)
        previo = msf[(msf.mes == m - 1) & msf.permno.isin(miembros)][["permno", "mcap"]]
        actual = msf[(msf.mes == m) & msf.permno.isin(miembros)][["permno", "ret"]]
        x = previo.merge(actual, on="permno").dropna()
        if len(x) == 0 or x.mcap.sum() <= 0:
            salida[m] = np.nan
            continue
        salida[m] = np.average(x.ret, weights=x.mcap) if ponderado else x.ret.mean()
    return pd.Series(salida)

## 8. El backtest: las dos versiones a la vez
Tarda varios minutos. Verás una línea por año.

In [ ]:
def backtest_comparado(top_n=TOP_N, roc_metodo=ROC_METODO):
    mensual, carteras, solapes = [], [], {}
    for anio in ANIOS:
        fecha = pd.Timestamp(f"{anio}-01-01")
        if not meses_del_anio(anio):
            break
        rk = {m: ranking(fecha, roc_metodo, m) for m in ["ltm", "fwd"]}
        top = {m: rk[m].head(top_n) for m in rk}
        for m in top:
            carteras.append(top[m].assign(anio=anio, version=m)[["anio", "version", "puesto", "conm", "ey", "roc"]])
        solapes[anio] = len(set(top["ltm"].gvkey) & set(top["fwd"].gvkey))
        tabla = pd.DataFrame({
            "mf_ltm": rentabilidad_cartera(list(top["ltm"].permno), anio),
            "mf_fwd": rentabilidad_cartera(list(top["fwd"].permno), anio) if len(top["fwd"]) else np.nan,
            "sp500": rentabilidad_indice(anio, ponderado=True),
            "sp500_igual_peso": rentabilidad_indice(anio, ponderado=False),
        })
        mensual.append(tabla)
        r = (1 + tabla).prod() - 1
        print(f"{anio}: original {r.mf_ltm:+7.1%} | forward {r.mf_fwd:+7.1%} | {NOMBRE_INDICE} {r.sp500:+7.1%} | "
              f"igual peso {r.sp500_igual_peso:+7.1%} | comunes {solapes[anio]}/{top_n}")
    return pd.concat(mensual), pd.concat(carteras), pd.Series(solapes)


mensual, carteras, solapes = backtest_comparado()


## 9. Resultados

In [ ]:
def metricas(serie):
    serie = serie.dropna()
    anios = len(serie) / 12
    curva = (1 + serie).cumprod()
    return pd.Series({"CAGR": curva.iloc[-1] ** (1 / anios) - 1, "Volatilidad": serie.std() * np.sqrt(12),
                      "Maxima caida": (curva / curva.cummax() - 1).min(), "Valor de 10.000 $": 10_000 * curva.iloc[-1]})


anual = mensual.groupby(mensual.index.year).apply(lambda x: (1 + x).prod() - 1)
resumen = mensual.apply(metricas).T
for c in ["mf_ltm", "mf_fwd"]:
    resumen.loc[c, "Anios por encima del indice"] = (anual[c] > anual.sp500).sum()
    resumen.loc[c, "Anios por encima del igual peso"] = (anual[c] > anual.sp500_igual_peso).sum()
completos = mensual[mensual.index.year <= 2025]
periodos = {f"{a}-{b}": completos[(completos.index.year >= a) & (completos.index.year <= b)].apply(lambda s: metricas(s)["CAGR"])
            for a, b in [(2001, 2007), (2008, 2015), (2016, 2025), (2001, 2025), (2006, 2025), (2015, 2025)]}
display(anual.style.format("{:+.1%}"))
display(resumen)
display(pd.DataFrame(periodos).T.style.format("{:.1%}"))


## 9 bis. Magic Formula rolling (compra por tercios) y tiempo hasta que el mercado reconoce la calidad
Cada 1 de enero se compran las **10 primeras** del ranking y se mantienen **3 años**: la cartera tiene siempre tres tramos (los comprados este año, el anterior y el de hace dos), el mismo dinero en cada tramo. Además se mide la rentabilidad de cada cartera anual en su **1.er, 2.º y 3.er año** frente al índice: si el exceso sigue en los años 2 y 3, el mercado tarda más de un año en reconocer la calidad.

In [ ]:
N_TRAMO = 10
cohortes = {m: {} for m in ["ltm", "fwd"]}
for anio in ANIOS:
    fecha = pd.Timestamp(f"{anio}-01-01")
    if not meses_del_anio(anio):
        break
    for m in cohortes:
        rk = ranking(fecha, ROC_METODO, m)
        cohortes[m][anio] = list(rk.head(N_TRAMO).permno)

rolling = []
for anio in sorted(cohortes["ltm"]):
    fila = {}
    for m in cohortes:
        tramos = [cohortes[m][a] for a in (anio, anio - 1, anio - 2) if cohortes[m].get(a)]
        if tramos:
            fila[f"rolling_{m}"] = pd.concat([rentabilidad_cartera(t, anio) for t in tramos], axis=1).mean(axis=1)
    rolling.append(pd.DataFrame(fila).reindex(mensual.loc[mensual.index.year == anio].index))
rolling = pd.concat(rolling)
mensual = mensual.join(rolling)
anual = mensual.groupby(mensual.index.year).apply(lambda x: (1 + x).prod() - 1)
display(mensual[["mf_ltm", "rolling_ltm", "mf_fwd", "rolling_fwd", "sp500", "sp500_igual_peso"]].apply(metricas).T)

# Rentabilidad de cada cartera de 30 en su año 1, 2 y 3 frente al índice
indices = anual[["sp500", "sp500_igual_peso"]]
evento = []
for anio in ANIOS:
    fecha = pd.Timestamp(f"{anio}-01-01")
    if not meses_del_anio(anio):
        break
    top = list(ranking(fecha, ROC_METODO, "ltm").head(TOP_N).permno)
    for h in (0, 1, 2):
        a = anio + h
        if a in indices.index:
            r = (1 + rentabilidad_cartera(top, a)).prod() - 1
            evento.append({"formada": anio, "anio_tenencia": h + 1, "ret": r,
                           "exceso_sp500": r - indices.loc[a, "sp500"], "exceso_igual_peso": r - indices.loc[a, "sp500_igual_peso"]})
evento = pd.DataFrame(evento)
display(evento.groupby("anio_tenencia")[["ret", "exceso_sp500", "exceso_igual_peso"]].mean().style.format("{:+.1%}"))


## 10. ¿Aciertan los analistas?
Se compara el EBIT previsto a 31 de diciembre con el EBIT de 12 meses que la empresa publicó después (aproximadamente el ejercicio siguiente).

In [ ]:
errores = []
for anio in range(2006, 2025):
    rk = ranking(pd.Timestamp(f"{anio}-01-01"), modo="fwd")
    real = foto_fundamental(pd.Timestamp(f"{anio + 1}-04-15")).set_index("gvkey").ebit
    rk["real"] = rk.gvkey.map(real)
    rk["error"] = rk.real / rk.ebit - 1
    rk["top"] = rk.puesto <= TOP_N
    errores.append(rk[["gvkey", "top", "error"]].assign(anio=anio))
errores = pd.concat(errores).replace([np.inf, -np.inf], np.nan).dropna()
tabla_err = errores.groupby("top").error.agg(mediana="median", pct_por_debajo=lambda x: (x < 0).mean(), n="count")
tabla_err.index = tabla_err.index.map({False: "Resto del universo", True: f"Top {TOP_N} forward"})
display(tabla_err)


## 10 bis. Panel de empresas para el análisis completo
Para cada año y cada empresa elegible: capitalización, EV, EBIT pasado y previsto, capital invertido, EBIT que publicó después y su rentabilidad del año siguiente. Es la base para comparar EE. UU. con Europa. Tarda unos minutos.

In [ ]:
def rentab_anual(permnos, anio):
    meses = meses_del_anio(anio)
    r = (msf[msf.permno.isin(permnos) & msf.mes.isin(meses)]
         .pivot_table(index="mes", columns="permno", values="ret").reindex(index=meses))
    return (1 + r.fillna(0)).prod() - 1

panel = []
for anio in ANIOS:
    fecha = pd.Timestamp(f"{anio}-01-01")
    if not meses_del_anio(anio):
        break
    rk = ranking(fecha, modo="ltm")
    rk["ebit_fwd"] = rk.gvkey.map(ebit_previsto_en(fecha))
    ff = foto_fundamental(pd.Timestamp(f"{anio + 1}-04-15"))
    real = ff.set_index("gvkey").ebit if len(ff) else pd.Series(dtype=float)
    rk["ebit_real_sig"] = rk.gvkey.map(real)
    rk["ret_anio"] = rk.permno.map(rentab_anual(list(rk.permno), anio))
    rk["anio"] = anio
    panel.append(rk[["anio", "gvkey", "permno", "conm", "gsector", "mcap", "ev", "ebit", "ebit_fwd", "ebit_real_sig",
                     "capital", "ey", "roc", "puesto", "ret_anio"]])
    print(anio, len(rk))
panel = pd.concat(panel)
print(f"Panel: {len(panel):,} filas, {panel.gvkey.nunique():,} empresas")


## 11. Guardar los resultados agregados
Descarga los archivos de la carpeta `resultados_forward` (botón derecho → *Download*) y pásamelos.

In [ ]:
import os
os.makedirs("resultados_forward", exist_ok=True)
mensual.rename_axis("mes").to_csv("resultados_forward/rentabilidad_mensual.csv")
anual.rename_axis("anio").to_csv("resultados_forward/rentabilidad_anual.csv")
resumen.to_csv("resultados_forward/resumen.csv")
pd.DataFrame(periodos).T.to_csv("resultados_forward/periodos.csv")
carteras.to_csv("resultados_forward/carteras.csv", index=False)
solapes.rename("comunes").to_csv("resultados_forward/solapes.csv")
tabla_err.to_csv("resultados_forward/error_analistas.csv")
panel.to_csv("resultados_forward/panel_eeuu.csv", index=False)
evento.to_csv("resultados_forward/tiempo_reconocimiento.csv", index=False)
mensual.rename_axis("mes").to_csv("resultados_forward/rentabilidad_mensual.csv")
anual.rename_axis("anio").to_csv("resultados_forward/rentabilidad_anual.csv")
print("Guardado:", os.listdir("resultados_forward"))
